# Funktionen für Parsen, Entitäten und Namespaces


Die Datei lb-test.xml ist eine einfache heiEditions-XML-Datei. Da werden Entites verwendet, die in https://digi.ub.uni-heidelberg.de/schema/tei/heiEDITIONS/declarations/heieditions-entities.txt" definiert sind. Im nächsten Code-Abschnitt kann man den Inhalt der Datei sehen.

In [1]:
example_file_path = 'beispiele/beispiel_data/lb-test.xml'
import codecs
with codecs.open(example_file_path, 'r', 'utf-8') as example_file:
    example = example_file.read()
    print(example)

<?xml version='1.0' encoding='UTF-8'?>
<?xml-model href="https://digi.ub.uni-heidelberg.de/schema/tei/heiEDITIONS/tei_hes.rng" type="application/xml" schematypens="http://relaxng.org/ns/structure/1.0"?><?xml-model href="https://digi.ub.uni-heidelberg.de/schema/tei/heiEDITIONS/tei_hes.rng" type="application/xml" schematypens="http://purl.oclc.org/dsdl/schematron"?>
<!DOCTYPE TEI SYSTEM "https://digi.ub.uni-heidelberg.de/schema/tei/heiEDITIONS/declarations/heieditions-entities.txt">
<TEI xmlns="http://www.tei-c.org/ns/1.0" xmlns:hei="https://digi.ub.uni-heidelberg.de/schema/tei/heiEDITIONS">
  <teiHeader>  
    <fileDesc>
      <titleStmt>
        <title ana="hc:MainTitle">Test Notes</title>
      </titleStmt>
      <publicationStmt>
        <p></p>
      </publicationStmt>
      <sourceDesc>
        <p></p>
      </sourceDesc>
    </fileDesc>
  </teiHeader>
  <facsimile>
    <surface ana="hc:Page" n="1r" xml:id="_1r">
      <zone ana="hc:VerticalFloatContainer">
        <zone ana="hc:Te

Wenn wir so eine Datei mit lxml/etree parsen wollen, kommt eine Fehlermeldung, da die externe Entities nicht geladen werden können.

In [2]:
from lxml import etree as et

try:
    tree = et.parse('beispiele/beispiel_data/lb-test.xml')
except SyntaxError as e:
    print(e)
    pass

Entity 'vsup' not defined, line 94, column 53 (lb-test.xml, line 94)


heipy bietet ein eigenes Parser, das diese Datei parsen kann.

In [3]:
from heipy.parsers import HeiEditionsParser

heiparser = HeiEditionsParser()
tree = et.parse('beispiele/beispiel_data/lb-test.xml', parser= heiparser)
root = tree.getroot()
print(root)

<Element {http://www.tei-c.org/ns/1.0}TEI at 0x7a6da15c1440>


Wenn wir in einer TEI solchen Datei XPath verwenden wollen, müssen wir entweder die Namespaces in geschweiften Klammern schreiben oder die Präfixe definieren. Also:

In [4]:
facsimile = root.findall('.//{http://www.tei-c.org/ns/1.0}facsimile')
facsimile = root.findall('.//tei:facsimile', namespaces= {'tei':'http://www.tei-c.org/ns/1.0'})

Die wichtigsten Präfixe (tei,xml,hei,hc,page,mets) werden in heipy schon in einer Variabel definiert, die importiert werden kann.

In [5]:
from heipy.namespaces import ns 

facsimile = root.findall('.//tei:facsimile', namespaces= ns)
print(facsimile)

[<Element {http://www.tei-c.org/ns/1.0}facsimile at 0x7a6da01b5880>]


Manchmal müssen wir mit lxml auch diese Präfixe vor dem Elementname schreiben. Das können wir auch mit der Funktion `prefix_format` aus heipy machen. Hier ein Beispiel, wenn wir ein neues `<link>` Element in TEI-Namespace mit etree erzeugen wollen oder alle xml:id von `<l>` Elemente suchen.

In [6]:
from heipy.namespaces import prefix_format

new_elelement = et.Element(prefix_format('tei','link'))

for line in root.findall('.//tei:l', namespaces=ns):
    print(line.attrib)
    line_id = line.get(prefix_format('xml','id'))
    altn = line.get(prefix_format('hei','altN'))
    print(line_id, altn)

{}
None None
{'n': '136', '{http://www.w3.org/XML/1998/namespace}id': 'l_136', '{https://digi.ub.uni-heidelberg.de/schema/tei/heiEDITIONS}altN': '134'}
l_136 134
{'n': '137', '{http://www.w3.org/XML/1998/namespace}id': 'l_137', '{https://digi.ub.uni-heidelberg.de/schema/tei/heiEDITIONS}altN': '135'}
l_137 135


# Transformations-Pipeline

Eine Pipeline besteht aus eine Serie von Schritten, die nacheinander ausgeführt werden. Es gibt unterschiedliche Arten von Schritten: XsltStep, PythonStep, AddAttribute, ValidationStep, DeleteStep. Diese Klassen sind in heipy.heipipe.steps definiert.

In [7]:
from heipy.heipipe.steps import Pipeline, XsltStep

pipe = Pipeline(name='Example_Pipe')

schritt1 = XsltStep(['src/heipy/heipipe/xslt/text_initials.xsl'], name="Initials")
pipe.add_step(schritt1)

schritt2 = XsltStep(['src/heipy/heipipe/xslt/text_markNoteAsEditorial.xsl'], name="Mark_note_as_editorial",
                    parameters= [{'note_classes': "hc:Comment"}])
pipe.add_step(schritt2)

result = pipe.execute(example_file_path)



Starting Pipeline Example_Pipe for beispiele/beispiel_data/lb-test.xml


Es gibt unterschiedliche Arten von Schritten: XsltStep, AddAttribute, DeleteStep, UnwrapStep, ValidationStep

In [8]:
from heipy.heipipe.steps import PythonStep, AddAttribute, ValidationStep, DeleteStep, UnwrapStep

pipe.add_step( DeleteStep(['facsimile']) )
# help(DeleteStep)

# pipe.add_step(ValidationStep())
# help(ValidationStep)

pipe.add_step(AddAttribute('//tei:title', 'ana', 'hc:MainTitle'))
# help(AddAttribute)

pipe.add_step(UnwrapStep([{'element_name': 'w'}]))
# help(UnwrapStep)


A PythonStep is the most complex kind of step. It requires a function that takes as a parameter a root from an xml object element and a parameters argument that can be empty. It must return the edited root. For example:

In [9]:
def add_ptr_after_l(root, parameters):
    ls = root.findall('.//tei:l', namespaces=ns)
    for l in ls:
        et.SubElement(l, prefix_format('tei','ptr'), nsmap=ns)
    return root

pipe.add_step(PythonStep(add_ptr_after_l))

pipe.execute(example_file_path)


Starting Pipeline Example_Pipe for beispiele/beispiel_data/lb-test.xml


'<TEI xmlns="http://www.tei-c.org/ns/1.0" xmlns:hei="https://digi.ub.uni-heidelberg.de/schema/tei/heiEDITIONS">\n  <teiHeader>  \n    <fileDesc>\n      <titleStmt>\n        <title ana="hc:MainTitle">Test Notes</title>\n      </titleStmt>\n      <publicationStmt>\n        <p/>\n      </publicationStmt>\n      <sourceDesc>\n        <p/>\n      </sourceDesc>\n    </fileDesc>\n  </teiHeader>\n  <facsimile>\n    <surface ana="hc:Page" n="1r" xml:id="_1r">\n      <zone ana="hc:VerticalFloatContainer">\n        <zone ana="hc:TextZone hc:Column" n="1ra" xml:id="_1ra"/>\n        <zone ana="hc:TextZone hc:Column" n="1rb" xml:id="_1rb"/>\n      </zone>\n    </surface>\n    <surface ana="hc:Page" xml:id="_1v" n="1v">\n      <zone ana="hc:TextZone hc:Column" n="1va" xml:id="_1va"/>\n      <zone ana="hc:TextZone hc:MarginalZone" xml:id="_1v_m1"/>\n      <zone ana="hc:TextZone hc:MarginalZone" xml:id="_1v_m2">\n          \n      </zone>\n    </surface>\n    <surface ana="hc:Page" xml:id="_2r" n="2r">

Da die meisten Pipelines immer sehr ähnlich sind, werden default-Pipelines in heipy definiert. Zur Zeit gibt es semantic und sourceDoc

In [10]:
from heipy.heipipe.pipeline_library.sourcedoc import SourceDocPipe
from heipy.heipipe.pipeline_library.semantic import SemanticPipe

semantic_pipe = SemanticPipe()
sourcedoc_pipe = SourceDocPipe()

Die einzelnen Schritten der Default-Pipelines werden in heipy.heipipe.step_library definiert. Man kann diese mit dem get_steps() Funktion auflisten.

In [11]:
for i, step in enumerate(sourcedoc_pipe.get_steps()):
    print(f'{i}: {step}')

0: XSLStep »initials« containing 1 transformations ['text_initials.xsl'] and 0 parameters .
1: XSLStep »transcription_note« containing 1 transformations ['text_transcriptionNote.xsl'] and 0 parameters .
2: XSLStep »connect_lb_and_segment« containing 3 transformations ['text_connectLbWithZone.xsl', 'text_moveIncludedInZone.xsl', 'text_connectSegmentWithLine.xsl'] and 0 parameters .
3: Python step »move_physical_beginnings«, using: <function move_physical_beginnings at 0x7a6da0040f40>
4: XSLStep »Whitespaces« containing 4 transformations ['text_trimWhitespaceAdjacentToPhysicalBeginnings.xsl', 'text_normalizeWhitespaceInMixedContent.xsl', 'text_stripWhitespaceInElementsStatedBySchema.xsl', 'text_normalizeWhitespaceInTokenizedContent.xsl'] and 0 parameters .
5: XSLStep »mark_note_as_editorial« containing 1 transformations ['text_markNoteAsEditorial.xsl'] and 1 parameters [{'note_classes': 'hc:TextCriticalNote hc:TranscriptionNote hc:TextConstitutionNote hc:Comment hc:FontesNote hc:VariantN

Man kann neue Schritte hinzufügen mit add_step(). Per Default am Ende der Pipeline, aber mit möglich index (s. oben um die Indexes zu sehen). Alternativ kann man after_step() oder before_step() verwenden

In [12]:
sourcedoc_pipe = SourceDocPipe()

sourcedoc_pipe.add_step(XsltStep(files=['pipelines/local_transformations/editorial_pc.xsl'], name="new_step_1"))

sourcedoc_pipe.add_step(XsltStep(files=['pipelines/local_transformations/editorial_pc.xsl'], name="new_step_2"), 
                        at_index= 0)
sourcedoc_pipe.add_step(XsltStep(files=['pipelines/local_transformations/editorial_pc.xsl'], name="new_step_3"), 
                        after_step='Whitespaces')
sourcedoc_pipe.add_step(XsltStep(files=['pipelines/local_transformations/editorial_pc.xsl'], name="new_step_4"), 
                        before_step='new_step_2')

sourcedoc_pipe.set_pipestep_parameter('mark_note_as_editorial','note_classes', 'hc:Comment')

for i, step in enumerate(sourcedoc_pipe.get_steps()):
    print(f'{i}: {step}')



0: XSLStep »new_step_4« containing 1 transformations ['pipelines/local_transformations/editorial_pc.xsl'] and 0 parameters .
1: XSLStep »new_step_2« containing 1 transformations ['pipelines/local_transformations/editorial_pc.xsl'] and 0 parameters .
2: XSLStep »initials« containing 1 transformations ['text_initials.xsl'] and 0 parameters .
3: XSLStep »transcription_note« containing 1 transformations ['text_transcriptionNote.xsl'] and 0 parameters .
4: XSLStep »connect_lb_and_segment« containing 3 transformations ['text_connectLbWithZone.xsl', 'text_moveIncludedInZone.xsl', 'text_connectSegmentWithLine.xsl'] and 0 parameters .
5: Python step »move_physical_beginnings«, using: <function move_physical_beginnings at 0x7a6da0040f40>
6: XSLStep »Whitespaces« containing 4 transformations ['text_trimWhitespaceAdjacentToPhysicalBeginnings.xsl', 'text_normalizeWhitespaceInMixedContent.xsl', 'text_stripWhitespaceInElementsStatedBySchema.xsl', 'text_normalizeWhitespaceInTokenizedContent.xsl'] and 

# Weitere Funktionen

Alle Charaktere in einem String auflisten

In [ ]:
from heipy.other import list_all_characters

example = "This is an exāple!"

list_all_characters(example)